In [19]:
!pip uninstall -y transformers optimum airllm
!pip install transformers==4.18 optimum==1.16.2 
!pip install airllm

  Using cached transformers-4.18.0-py3-none-any.whl.metadata (70 kB)
  Using cached optimum-1.16.2-py3-none-any.whl.metadata (17 kB)
  Using cached sacremoses-0.1.1-py3-none-any.whl.metadata (8.3 kB)
  Using cached tokenizers-0.12.1.tar.gz (220 kB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
INFO: pip is looking at multiple versions of optimum to determine which version is compatible with other requirements. This could take a while.
ERROR: Cannot install optimum==1.16.2 and transformers==4.18 because these package versions have conflicting dependencies.

The conflict is caused by:
    The user requested transformers==4.18
    optimum 1.16.2 depends on transformers>=4.26.0

Additionally, some packages in these conflicts have no matching distributions available for your environment:
    transformers

To fix this you could try to:
1. loosen the range of package versions you've specified
2. remove pa

# AirLLM CPU Chat Test
This notebook demonstrates running a large language model on CPU using AirLLM's layer-by-layer inference technique, which reduces memory requirements significantly.

In [20]:
import os
import torch
from airllm import AirLLMLlama2

# Configuration
MODEL_ID = "meta-llama/Llama-2-7b-chat-hf"  # Change to any supported model
MAX_LENGTH = 512
MAX_NEW_TOKENS = 200

# Force CPU usage
os.environ["CUDA_VISIBLE_DEVICES"] = ""

print(f"PyTorch version: {torch.__version__}")
print(f"Using device: CPU")
print(f"Model: {MODEL_ID}")

ModuleNotFoundError: No module named 'optimum.bettertransformer'

In [ ]:
# Initialize the model on CPU
# AirLLM will split the model into chunks and load them sequentially
print("Loading model... This may take a few minutes on first run.")

model = AirLLMLlama2(
    MODEL_ID,
    device='cpu',
    dtype=torch.float32,  # Use float32 for CPU compatibility
    compression='4bit'    # Use 4-bit compression to reduce memory usage
)

print("Model loaded successfully!")

In [ ]:
from transformers import AutoTokenizer

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

print("Tokenizer loaded successfully!")

In [ ]:
def format_prompt(user_message: str, system_prompt: str = None) -> str:
    """Format message in Llama-2 chat format."""
    if system_prompt is None:
        system_prompt = "You are a helpful, respectful and honest assistant."
    
    prompt = f"""<s>[INST] <<SYS>>
{system_prompt}
<</SYS>>

{user_message} [/INST]"""
    return prompt


def generate_response(user_message: str, system_prompt: str = None) -> str:
    """Generate a response from the model given a user message."""
    # Format the prompt
    prompt = format_prompt(user_message, system_prompt)
    
    # Tokenize
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        max_length=MAX_LENGTH,
        truncation=True,
        padding=True
    )
    
    print(f"Input tokens: {inputs['input_ids'].shape[1]}")
    print("Generating response...")
    
    # Generate
    with torch.no_grad():
        output_tokens = model.generate(
            inputs['input_ids'],
            max_new_tokens=MAX_NEW_TOKENS,
            use_cache=False,  # Required for AirLLM
            return_dict_in_generate=False
        )
    
    # Decode only the new tokens
    new_tokens = output_tokens[0][inputs['input_ids'].shape[1]:]
    response = tokenizer.decode(new_tokens, skip_special_tokens=True)
    
    return response.strip()


print("Chat functions defined successfully!")

In [ ]:
# === Single Query Test ===
import time

test_message = "What is the capital of France? Please answer briefly."

print(f"User: {test_message}")
print("-" * 50)

start_time = time.time()
response = generate_response(test_message)
elapsed = time.time() - start_time

print(f"Assistant: {response}")
print("-" * 50)
print(f"Generation time: {elapsed:.2f} seconds")

In [ ]:
# === Multi-turn Chat Test ===
chat_history = []

def chat(user_message: str, system_prompt: str = None) -> str:
    """Simple chat function that maintains conversation history."""
    print(f"\n👤 User: {user_message}")
    
    start = time.time()
    response = generate_response(user_message, system_prompt)
    elapsed = time.time() - start
    
    chat_history.append({"user": user_message, "assistant": response})
    
    print(f"🤖 Assistant: {response}")
    print(f"   ⏱️  {elapsed:.2f}s")
    
    return response


# Run a series of test prompts
test_prompts = [
    "Hello! Can you introduce yourself briefly?",
    "What are the benefits of running AI models on CPU?",
    "Give me a one-sentence summary of machine learning."
]

system_prompt = "You are a concise AI assistant. Keep your answers short and to the point."

print("=" * 60)
print("CHAT TEST SESSION")
print("=" * 60)

for prompt in test_prompts:
    chat(prompt, system_prompt)
    print()

In [ ]:
# === Interactive Chat Loop ===
# Run this cell for an interactive chat session

print("=" * 60)
print("INTERACTIVE CHAT (type 'quit' to exit)")
print("=" * 60)

system_prompt = "You are a helpful AI assistant."

while True:
    try:
        user_input = input("\nYou: ").strip()
        
        if not user_input:
            continue
        
        if user_input.lower() in ['quit', 'exit', 'bye']:
            print("Goodbye!")
            break
        
        chat(user_input, system_prompt)
        
    except KeyboardInterrupt:
        print("\nSession interrupted.")
        break

In [ ]:
# === Session Summary ===
print("=" * 60)
print("CHAT HISTORY SUMMARY")
print("=" * 60)

for i, exchange in enumerate(chat_history, 1):
    print(f"\n[Turn {i}]")
    print(f"  User     : {exchange['user']}")
    print(f"  Assistant: {exchange['assistant']}")

print(f"\nTotal turns: {len(chat_history)}")